In [ ]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

cwd = Path.cwd().resolve()
workspace_root = cwd
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from cert_tools.linalg_tools import rank_project
from cert_tools.sdp_solvers import solve_sdp
from plot_poly import (
    plot_calibration_setup,
    plot_cost,
    plot_global,
    plot_problem_matrices,
    polish_cost_figure,
)
from popcor.popcor.examples import Poly4Lifter, Poly6Lifter
from popcor.popcor.examples.rotation_lifter import RotationLifter

np.random.seed(42)  # Fix seed for reproducibility
plt.rcParams.update({"font.size": 12})
%matplotlib inline

# HW3.3 - Convex Relaxations [30 pts]

## Overview

This bonus homework explores **semidefinite (SDP) relaxations**, a technique for finding global optima of hard nonconvex problems. Instead of solving a problem directly, we relax it to a convex SDP that can be solved efficiently.

**Key idea:** Lift the problem to a higher-dimensional space using a **moment matrix**, solve the lifted problem, and measure quality by comparing the SDP bound to the true optimum.

**Your task:** Compare this strategy for different problems, and understand how to analyze performance.

## Part a: Relaxations for Polynomial Problems [10 pts]

We provide the following 3 test problems.
- **Poly4 Type A**: Degree-4 polynomial with one global minimum, one local minimum
- **Poly4 Type B**: Degree-4 polynomial with one two global minima
- **Poly6 Type A**: Degree-6 polynomial with one global minimum, two local minima

**3.a.1**: For each problem, we want to compute: [5 pts]
1. The SDP relaxation value (lower bound)
2. The exact global minimum cost
3. Relaxation gap = (SDP value - exact minimum)
4. Rank of the SDP solution matrix X and the optimal solution if it is rank one. 

In below's code, fill in the missing parts. 


In [ ]:
import pandas as pd

lifter_configs = {
    "Poly4_TypeA": {
        "lifter_class": Poly4Lifter,
        "params": {"example_type": "A"},
        "add_redundant": False,
        "xlims": [-2.0, 3.1],
    },
    "Poly4_TypeB": {
        "lifter_class": Poly4Lifter,
        "params": {"example_type": "B"},
        "add_redundant": False,
        "xlims": [0.0, 4.0],
    },
    "Poly6_TypeA": {
        "lifter_class": Poly6Lifter,
        "params": {"example_type": "A"},
        "add_redundant": False,
        "xlims": [-2.0, 5.1],
    },
    "Poly4_TypeA_with_redundant": {
        "lifter_class": Poly4Lifter,
        "params": {"example_type": "A"},
        "add_redundant": True,
        "xlims": [-2.0, 3.1],
    },
    "Poly4_TypeB_with_redundant": {
        "lifter_class": Poly4Lifter,
        "params": {"example_type": "B"},
        "add_redundant": True,
        "xlims": [0.0, 4.0],
    },
    "Poly6_TypeA_with_redundant": {
        "lifter_class": Poly6Lifter,
        "params": {"example_type": "A"},
        "add_redundant": True,
        "xlims": [-2.0, 5.1],
    },
}

results = {}

for lifter_name, config in lifter_configs.items():
    print(f"\n{'='*60}")
    print(f"Processing: {lifter_name}")
    print(f"{'='*60}")

    # 1. Instantiate and plot the lifter
    lifter = config["lifter_class"].create_example(**config["params"] )
    fig, ax, line = lifter.plot_cost(xlims=config["xlims"] )
    ax.set_title(f"{lifter_name}")

    # 2. Get Q matrix and constraints
    Q = lifter.get_Q()
    A_0, b_0 = lifter.get_A0()
    A_known, b_known = lifter.get_A_known(add_redundant=config["add_redundant"] )

    # 3. Create constraints and solve the SDP
    # ================ Fill in below ====================================
    # Solve the SDP relaxation using solve_sdp from cert_tools.sdp_solvers

    X_opt = None
    # ====================================================

    # ================ Fill in below ====================================
    # Extract the cost from the relaxed problem and the original cost and compute the gpa.

    sdp_value = None
    exact_value = None
    gap = None
    # ====================================================
    ax.axhline(sdp_value, color="C2", linestyle="--", label=f"SDP Value: {sdp_value:.4f}")
    ax.axhline(exact_value, color="C0", linestyle="-.", label=f"Exact Minimum: {exact_value:.4f}")

    # 4. Extract rank gap
    # ================ Fill in below ====================================
    # - Extract the rank-1 projection from X_opt using rank_project and get the eigenvalue ration (EVR) from the info returned by rank_project.
    # - Extract the solution by getting the right element from X_opt.

    evr = None
    x_opt_round = None
    x_opt_pick = None
    # ====================================================
    ax.scatter(x_opt_pick, lifter.get_cost(x_opt_pick), color="C3", marker="x", s=100, label=f"x picked")
    ax.scatter(x_opt_round, lifter.get_cost(x_opt_round), color="C1", marker="x", s=100, label=f"x round")

    # 5. Store results
    results[lifter_name] = {
        "SDP_value": sdp_value,
        "exact_minimum": exact_value,
        "gap": gap,
        "EVR": evr,
        "add_redundant": config["add_redundant"],
    }

# Print summary table
df = pd.DataFrame(results).T
print("\n" + "=" * 80)
print("SUMMARY TABLE:")
print("=" * 80)
print(df.to_string())

**3.a.2** Answer the following questions: [5 pts] 
- Which problems give **tight** relaxations, in terms of the cost gap? Do they all provide valid lower bounds? 
- Which problems allow to retrieve **tight** relaxations, in terms of being able to retrieve the true global minimum? Explain why that is the case. 
- How expensive is solving the SDP compared to solving the problem locally? 

**Answer**: 

## Part b: Rotation Examples [10 pts]

Now we implement the convex relaxations for more challenging examples that involve a rotation variable. 

We introduce the unknown vector x which is composed of the homogenization variable and the vectorized rotation matrix: 

$x = [1, R_{vec}]^T$ where $R_{vec}$ is the vectorized (flattened) version of the rotation matrix $R$.

We have two different problems: 
- **RotationLifter Type A**: Two global minima (symmetric case)
- **RotationLifter Type B**: One global minimum and one local minimum

We generated the configurations that allow you to create these two examples, using the same logic as in the example above. 

**3.b.1** Copy the code from above to find the convex relaxations for those two rotation lifters. You can use the same functions (get_A_known, get_Q, etc.) as above. [5pts]

In [ ]:
# Create Rotation Lifter (same lifter, two different examples)
from popcor.popcor.examples.rotation_lifter import RotationLifter

lifter_configs = {
    "Rotation_TypeA": {
        "lifter_class": RotationLifter,
        "params": {"example_type": "A"},
        "add_redundant": False,
    },
    "Rotation_TypeB": {
        "lifter_class": RotationLifter,
        "params": {"example_type": "B"},
        "add_redundant": False,
    },
    "Rotation_TypeA_with_redundant": {
        "lifter_class": RotationLifter,
        "params": {"example_type": "A"},
        "add_redundant": True,
    },
    "Rotation_TypeB_with_redundant": {
        "lifter_class": RotationLifter,
        "params": {"example_type": "B"},
        "add_redundant": True,
    },
}

results = {}

for lifter_name, config in lifter_configs.items():
    print(f"\n{'='*60}")
    print(f"Processing: {lifter_name}")
    print(f"{'='*60}")

    # ================ Fill in below ====================================
    # Extract sections 1-5 from above and apply to the rotation lifter examples.

    # =========================================

df_rot = pd.DataFrame(results).T
print("\n" + "=" * 80)
print("ROTATION PROBLEMS - SUMMARY TABLE:")
print("=" * 80)
print(df_rot.to_string())

**3.b.2** Answer the same questions as in 3.a.2: 

- Which problems give **tight** relaxations, in terms of the cost gap? Do they all provide valid lower bounds? 
- Which problems allow to retrieve **tight** relaxations, in terms of being able to retrieve the true global minimum? Explain why that is the case. 
- How expensive is solving the SDP compared to solving the problem locally? 

**Answer**: 

## Part 3.c: Simple Calibration Problem [10 pts]

We saw in Practical 2 a simple camera calibration problem. Here we are going to simplify even further. Assume that you have a camera with known translation $t=0$ and you want to calibrate its rotation based on some landmark measurments. 

The below code generates and plots a simulated calibration setup, where we show the ground-truth rotation matrix and the measured vs. reconstructed landmark positions.


In [ ]:
# Build a 2D calibration toy example so we can visualize everything clearly.
d = 2
n_landmarks = 10
angle_true = np.deg2rad(30.0)
R_true = np.array([[np.cos(angle_true), -np.sin(angle_true)],
                   [np.sin(angle_true),  np.cos(angle_true)]])

landmarks_world = np.random.randn(d, n_landmarks) * 1.5 + np.array([[2.0], [1.0]])
landmarks_true = R_true.T @ landmarks_world
noise_std = 0.05
landmarks_measured = landmarks_true + noise_std * np.random.randn(d, n_landmarks)

plot_calibration_setup(
    rotations={"ground truth": R_true},
    landmarks_world=landmarks_world,
    landmarks_measured=landmarks_measured,
    d=d,
    title="Calibration setup in world coordinates",
)


**3.c.1** Given the same cost function you used in Practical 2, but now with fixed translation, provide the code to populate matrix **Q**. [5 pts]

In [ ]:
# Cost function: sum of squared reprojection errors
# For each landmark i: ||measured_i - R^T * true_i||^2 = ||measured_i||^2 - 2*measured_i^T*R^T*true_i + ||R^T*true_i||^2
# After vectorization with x = [1, vec(R)], construct Q such that x^T*Q*x gives the cost

n_var = 1 + d * d  # 1 (homogenization) + d^2 (vectorized dxd rotation matrix)
Q = np.zeros((n_var, n_var))

# solution:
cost_manual = 0.0
I_d = np.eye(d)

for i in range(n_landmarks):
    measured_i = landmarks_measured[:, i]
    true_i = landmarks_world[:, i]

    ### ============ 3.c.1 Fill in below ============
    # construct Q. If you do it correctly, the code below should give you the 
    # same cost as cost_manual!
    Q += None
    ### ============================================


    cost_manual += np.linalg.norm(measured_i - R_true.T @ true_i) ** 2

Q = Q / n_landmarks  # Normalize by number of landmarks

print(f"d = {d}, n_landmarks = {n_landmarks}, noise_std = {noise_std}")
print(f"Q matrix shape: {Q.shape}")

# Validate Q against the known true rotation used to generate the synthetic data.
r_true = R_true.flatten(order="F")
x_true = np.hstack([1.0, r_true])
cost_from_Q = float(x_true.T @ Q @ x_true)
cost_manual = float(cost_manual / n_landmarks)

assert np.isclose(cost_from_Q, cost_manual, atol=1e-10)

**3.c.2** Solve the problem using a convex relaxation, plot the solution, and comment on the result. [5 pts]

In [ ]:
### ============ 3.c.2 Fill in below ============
# Solve the calibration objective with an SDP relaxation using the same flow as Part b.

R_hat = None
### ============================================

plot_calibration_setup(
    rotations={"ground truth": R_true, "SDP estimate": R_hat},
    landmarks_world=landmarks_world,
    landmarks_measured=landmarks_measured,
    d=d,
    title="Calibration result in world coordinates (GT vs SDP)",
)